# UFM-Rec — dataset2 All Beauty trên Colab T4

Code = bản **khóa hash của FITLAB**: 5 file train/extract bit-identical, sha256 khớp manifest `runs/second_dataset_plan_v1.json` (kiểm tra lại ngay trên Colab ở cell 3).
Protocol: validation-only (không đụng test), checkpoint theo đúng plan, marker `completed.json` đúng field mà suite FITLAB kiểm tra (`smoke_run=False`, `test_evaluated=False`).

**Trước khi chạy:**
1. Runtime → Change runtime type → **T4 GPU** (cần T4 cho CLIP + train).
2. File → Upload to session storage → chọn `src_scripts.tar.gz` (bản sao đã để sẵn ở Drive nếu phiên trước đã up).
3. Chạy tuần tự các cell; mỗi stage tự bỏ qua phần đã xong và tự đồng bộ lên Drive → phiên sau chạy lại là resume.


In [1]:
import os, sys, shutil, subprocess, json, hashlib, time, traceback
from pathlib import Path

def _diag(msg):
    for p in ['/content/diag.txt', '/content/drive/MyDrive/ufm_rec_all_beauty/diag.txt']:
        try:
            with open(p, 'a', encoding='utf-8') as f:
                f.write(msg + '
')
        except Exception:
            pass

_diag('=== CELL1 START ===')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    _diag('MOUNT_OK')
except Exception:
    _diag('MOUNT_FAIL:
' + traceback.format_exc())
    raise
DRIVE = Path('/content/drive/MyDrive/ufm_rec_all_beauty'); DRIVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/ufm_rec'); ROOT.mkdir(exist_ok=True)
(ROOT/'reports').mkdir(exist_ok=True); (ROOT/'tmp').mkdir(exist_ok=True)
_diag('PATHS_OK')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
_diag('GPU: ' + gpu)
if not gpu.startswith('Tesla T4'):
    print('WARNING: chua thay T4 - vao Runtime > Change runtime type > T4 GPU roi chay lai.')
pkgs = ['numpy==2.2.6','pandas==2.2.3','scipy==1.17.1','scikit-learn==1.6.1','joblib==1.6.0',
        'pillow==12.3.0','transformers==4.57.6','huggingface-hub==0.36.2']
_diag('PIP_START')
r = subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, capture_output=True, text=True)
_diag('PIP_RC=' + str(r.returncode))
_diag('PIP_STDERR_TAIL=' + (r.stderr or '')[-800:])
if r.returncode != 0:
    raise RuntimeError('pip install failed: ' + (r.stderr or '')[-800:])
import numpy, pandas, sklearn, transformers, torch
print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__,
      '| transformers', transformers.__version__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())
_diag('CELL1_DONE')


SyntaxError: unterminated string literal (detected at line 8) (449166474.py, line 8)

In [ ]:
print('===DIAG_READ===')
import os
for p in ['/content/diag.txt', '/content/drive/MyDrive/ufm_rec_all_beauty/diag.txt']:
    if os.path.exists(p):
        print('FILE:', p)
        print(open(p, encoding='utf-8').read()[-2500:])
print('===DIAG_END===')


===DIAG_READ===
===DIAG_END===


In [ ]:
tar = Path('/content/src_scripts.tar.gz')
if not (ROOT/'src'/'train_ufm_recommender.py').exists():
    if not tar.exists():
        tar = DRIVE/'src_scripts.tar.gz'
    assert tar.exists(), 'Thieu src_scripts.tar.gz (session storage hoac Drive)'
    subprocess.run(['tar','xzf',str(tar),'-C',str(ROOT)], check=True)
if tar.exists() and not (DRIVE/'src_scripts.tar.gz').exists():
    shutil.copy2(tar, DRIVE/'src_scripts.tar.gz')
print('src:', len(list((ROOT/'src').glob('*.py'))), '| scripts:', len(list((ROOT/'scripts').glob('*.py'))))

LOCKED = {
 'train_sasrec.py': '8c4d8ab6659b7ecb0ec536fe9dac6b79b0afd11246b1c4ab87664c5d57d02f8c',
 'comparison_models.py': '8f53f51fa114fee31f2969972eb8287d07cd94726658e85f38840cac311705f3',
 'train_ufm_recommender.py': 'd92d32a1c6c6b3c230976ccb3f8a1773b935df1b8dfed8c691c315ddaae05543',
 'ufm_model.py': '12c5b5f070ec0353b0a11e345acc2055b7a322aa6c952f982a8d3fba4adf0ba6',
 'extract_foundation_features.py': 'd0a72322b46935ed66ff07bce85da498612234a918d754677878f8b8229b9b74',
}
def sha256(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for c in iter(lambda: f.read(8*1024*1024), b''):
            h.update(c)
    return h.hexdigest()
for name, want in LOCKED.items():
    got = sha256(ROOT/'src'/name)
    assert got == want, f'{name}: {got} != {want}'
print('5 file khoa hash OK - bit-identical voi FITLAB')

def copy_tree(src, dst):
    src, dst = Path(src), Path(dst); dst.mkdir(parents=True, exist_ok=True)
    for p in src.rglob('*'):
        if p.is_file():
            q = dst/p.relative_to(src)
            if not q.exists() or q.stat().st_size != p.stat().st_size:
                q.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, q)

def sync_safe(src, dst, label):
    try:
        copy_tree(src, dst); print('synced ->', label)
    except Exception as e:
        print('SYNC FAIL', label, repr(e))

def reset_partial(d, marker):
    d = Path(d)
    if d.exists() and (marker is None or not (d/marker).exists()):
        shutil.rmtree(d); print('reset partial:', d)

def restore_if_missing(vm, drive_dir, marker):
    vm, drive_dir = Path(vm), Path(drive_dir)
    if (vm/marker).exists(): return
    if (drive_dir/marker).exists():
        print('restore from Drive:', vm); copy_tree(drive_dir, vm)

DATA   = ROOT/'data/processed/all_beauty_full_temporal'
LEGACY = ROOT/'data/processed/all_beauty_gpu_cache_h20_t64_v1'
GRAPH  = ROOT/'data/processed/all_beauty_positive_graph_h20_v1'
FEATURES = DATA/'foundation_clip_b32_v1'
DRIVE_DATA   = DRIVE/'all_beauty_full_temporal'
DRIVE_LEGACY = DRIVE/'all_beauty_gpu_cache_h20_t64_v1'
DRIVE_GRAPH  = DRIVE/'all_beauty_positive_graph_h20_v1'
DRIVE_FEAT   = DRIVE/'features_clip_b32_v1'
DRIVE_RAW    = DRIVE/'all_beauty_raw'
DRIVE_RUNS   = DRIVE/'runs'
print('paths OK')


In [ ]:
restore_if_missing(ROOT/'data/raw/all_beauty_2023', DRIVE_RAW, 'sources.json')
restore_if_missing(DATA, DRIVE_DATA, 'split_manifest.json')
restore_if_missing(DATA/'content', DRIVE_DATA/'content', 'content_report.json')
restore_if_missing(DATA/'evaluation', DRIVE_DATA/'evaluation', 'sampling_report.json')
restore_if_missing(DATA/'foundation_catalog_v1', DRIVE_DATA/'foundation_catalog_v1', 'catalog_report.json')
restore_if_missing(DATA/'models/content_baseline', DRIVE_DATA/'models/content_baseline', 'metrics.json')
restore_if_missing(LEGACY, DRIVE_LEGACY, 'complete.json')
restore_if_missing(GRAPH, DRIVE_GRAPH, 'complete.json')

if not (DATA/'split_manifest.json').exists():
    reset_partial(DATA, 'split_manifest.json')
    reset_partial(ROOT/'tmp/prepare_all_beauty_full_temporal', None)
    t0 = time.time()
    subprocess.run([sys.executable,'-u',str(ROOT/'scripts/prepare_second_dataset.py')],
                   cwd=ROOT, check=True)
    print('PREPARE done in', round((time.time()-t0)/60, 1), 'min')
else:
    print('preparation da co san')
sync_safe(ROOT/'data/raw/all_beauty_2023', DRIVE_RAW, 'raw')
sync_safe(DATA, DRIVE_DATA, 'data')
sync_safe(ROOT/'reports', DRIVE/'reports', 'reports')
print('PREPARE STAGE OK')


In [ ]:
if not (GRAPH/'complete.json').exists():
    subprocess.run([sys.executable,'-u',str(ROOT/'src/train_ufm_recommender.py'),'--mode','prepare',
                    '--data',str(DATA),'--legacy-cache',str(LEGACY),'--cache',str(GRAPH),'--history','20'],
                   cwd=ROOT, check=True)
sync_safe(GRAPH, DRIVE_GRAPH, 'graph')
sync_safe(LEGACY, DRIVE_LEGACY, 'legacy')
print('GRAPH STAGE OK')


In [ ]:
restore_if_missing(FEATURES, DRIVE_FEAT, 'complete.json')
if not (FEATURES/'complete.json').exists():
    cmd = [sys.executable,'-u',str(ROOT/'src/extract_foundation_features.py'),'--device','cuda',
           '--limit','0','--batch-size','64','--catalog',str(DATA/'foundation_catalog_v1'),
           '--output',str(FEATURES),'--model-cache',str(ROOT/'tmp/hf_ufm_models')]
    if (FEATURES/'progress.json').exists():
        cmd.append('--resume')
    t0 = time.time()
    subprocess.run(cmd, cwd=ROOT, check=True)
    print('CLIP done in', round((time.time()-t0)/60, 1), 'min')
marker = json.loads((FEATURES/'complete.json').read_text())
assert marker.get('limit') == 0 and marker.get('encoder_frozen') is True, marker
print('CLIP marker OK:', {k: marker.get(k) for k in ('limit','encoder_frozen')})
sync_safe(FEATURES, DRIVE_FEAT, 'features')
print('CLIP STAGE OK')


In [ ]:
COMMON = ['--history','20','--dim','128','--heads','4','--layers','2','--batch-size','256',
          '--negatives','8','--epochs','20','--lr','0.0003','--patience','3',
          '--checkpoint-every','1000','--eval-batch','64','--validation-cap','0','--threads','4']
JOBS = [
 dict(model='ufm',     seed=42,   dir='runs/all_beauty_ufm_full_s42_v1'),
 dict(model='sasrec',  seed=42,   dir='runs/all_beauty_sasrec_s42_v2'),
 dict(model='bert4rec',seed=42,   dir='runs/all_beauty_bert4rec_s42_v2'),
 dict(model='concat',  seed=42,   dir='runs/all_beauty_concat_s42_v2'),
 dict(model='sasrec',  seed=7,    dir='runs/all_beauty_sasrec_s7_v2'),
 dict(model='bert4rec',seed=7,    dir='runs/all_beauty_bert4rec_s7_v2'),
 dict(model='concat',  seed=7,    dir='runs/all_beauty_concat_s7_v2'),
 dict(model='ufm',     seed=7,    dir='runs/all_beauty_ufm_full_s7_v1'),
 dict(model='sasrec',  seed=2026, dir='runs/all_beauty_sasrec_s2026_v2'),
 dict(model='bert4rec',seed=2026, dir='runs/all_beauty_bert4rec_s2026_v2'),
 dict(model='concat',  seed=2026, dir='runs/all_beauty_concat_s2026_v2'),
 dict(model='ufm',     seed=2026, dir='runs/all_beauty_ufm_full_s2026_v1'),
]

def build_args(job):
    seed = str(job['seed'])
    if job['model'] == 'ufm':
        return ['--variant','full','--output',str(ROOT/job['dir']),'--cache',str(GRAPH),
                '--data',str(DATA),'--legacy-cache',str(LEGACY),'--features',str(FEATURES),
                *COMMON,'--seed',seed,'--weight-decay','0.0001','--dropout','0.1','--id-dropout','0.5',
                '--fixed-alpha','0.5','--lambda-align','0.01','--lambda-unc','0.01',
                '--lambda-cal','0.1','--amp']
    return ['--model',job['model'],'--output',str(ROOT/job['dir']),'--graph',str(GRAPH),
            '--data',str(DATA),'--legacy-cache',str(LEGACY),'--features',str(FEATURES),
            *COMMON,'--seed',seed]

def valid_marker(p):
    d = json.loads(p.read_text())
    return d.get('smoke_run') is False and d.get('test_evaluated') is False

def tail(p, n=160):
    with p.open('rb') as f:
        f.seek(0, 2); size = f.tell(); f.seek(max(0, size-8192))
        lines = f.read().decode(errors='replace').strip().splitlines()
    return lines[-1][:n] if lines else ''

BUDGET = 10.0*3600
t0 = time.time()
results = []
for job in JOBS:
    out = ROOT/job['dir']; marker = out/'completed.json'
    if marker.exists():
        if not valid_marker(marker):
            raise RuntimeError(f'Marker khong hop le: {job["dir"]}')
        score = json.loads(marker.read_text()).get('best_cold_macro_ndcg@10')
        results.append((job['dir'], 'skip', score))
        print('SKIP', job['dir'], flush=True); continue
    if time.time()-t0 > BUDGET:
        print('Het ngan sach phien - dung launch, chay lai cell nay o phien sau.', flush=True); break
    out.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable,'-u',
           str(ROOT/'src'/('train_ufm_recommender.py' if job['model']=='ufm' else 'train_sasrec.py'))]
    cmd += build_args(job)
    resume = (out/'latest.pt').exists()
    if resume:
        cmd.append('--resume')
    log = ROOT/'runs'/(out.name+'.log')
    print('LAUNCH', out.name, '| resume =', resume, flush=True)
    with log.open('ab') as f:
        child = subprocess.Popen(cmd, cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        while child.poll() is None:
            time.sleep(45)
            print('   ...', tail(log), flush=True)
    if child.returncode:
        raise RuntimeError(f'{out.name} exit {child.returncode} - xem log')
    if not valid_marker(marker):
        raise RuntimeError(f'Marker khong hop le: {job["dir"]}')
    score = json.loads(marker.read_text()).get('best_cold_macro_ndcg@10')
    try:
        for name in ('best.pt','config.json','completed.json','feature_audit.json'):
            p = out/name
            if p.exists():
                q = DRIVE_RUNS/out.name/name; q.parent.mkdir(parents=True, exist_ok=True)
                if not q.exists() or q.stat().st_size != p.stat().st_size:
                    shutil.copy2(p, q)
        shutil.copy2(log, DRIVE_RUNS/out.name/out.name+'.log')
        (out/'latest.pt').unlink(missing_ok=True)
    except Exception as e:
        print('SYNC FAIL run', out.name, repr(e))
    results.append((out.name, 'done', score))
    print('DONE', out.name, '| best_cold_macro_ndcg@10 =', score, flush=True)
print(json.dumps(results, indent=2))


In [ ]:
print('== TONG KET ==')
for name, status, score in results:
    print(f'{status:4s} {name:40s} {score}')
print()
print('Sau khi xong 12 job: best.pt + marker da nam o Drive/ufm_rec_all_beauty/runs/')
print('Buoc chuyen ve FITLAB se lam o phien rieng (copy vao cac path runs/all_beauty_*).')
